# 02 — GRPO Reinforcement Learning

This notebook applies **Group Relative Policy Optimization (GRPO)** on top of
the SFT-tuned model to improve mathematical reasoning.

**Pipeline:**
1. Merge the SFT LoRA adapter into the base model
2. Load the merged model with Unsloth (2x faster training)
3. Attach a fresh LoRA adapter for RL training
4. Train with 4 reward functions (format exact, format approx, correctness, number extraction)

**Dataset:** GSM8K (grade school math — good learning signal for GRPO)

**Requirements:** Google Colab with A100 GPU (High-RAM recommended).

## 1. Setup

In [ ]:
# Clone the repo (replace with your GitHub URL)
!git clone https://github.com/Roogard/math-rl-tuning.git
%cd math-rl-tuning

# Clean stale Unsloth compiled caches from previous runs
!rm -rf unsloth_compiled_cache

# Fix protobuf 5.x incompatibility (MessageFactory.GetPrototype removed)
!pip install "protobuf<5" --quiet

# Install unsloth for 2x faster training (no vLLM needed)
!pip install unsloth --quiet

# Install our package and remaining deps
!pip install -e . --quiet
!pip install bitsandbytes latex2sympy2 --quiet

In [ ]:
# Verify core imports — do NOT import unsloth here!
# Importing unsloth auto-patches TRL's GRPOTrainer with a buggy version
# for Mistral. Unsloth is imported lazily during model loading only.
import torch
from trl import GRPOTrainer, GRPOConfig
import trl
print(f"PyTorch: {torch.__version__}")
print(f"TRL:     {trl.__version__}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import importlib.util
has_unsloth = importlib.util.find_spec("unsloth") is not None
print(f"Unsloth installed: {has_unsloth}")
print("All imports OK")

## 2. Configuration

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from math_rl_tuning.config import load_config
from math_rl_tuning.utils import setup_hf_token, setup_wandb, mount_google_drive

cfg = load_config()

# --- Authentication ---
setup_hf_token()
setup_wandb(cfg.grpo_training.report_to and "math-rl-grpo")

# Mount Google Drive (to load SFT adapter and save RL model)
mount_google_drive()

## 3. (Optional) Customize Config

In [ ]:
import os

# Auto-detect SFT adapter path — check common locations
_candidates = [
    "/content/drive/MyDrive/math-rl-tuning/sft",  # Google Drive (from notebook 01)
    cfg.paths.sft_output_dir,                       # Local outputs/sft
    "./outputs/sft",                                # Relative fallback
]

SFT_ADAPTER_PATH = None
for path in _candidates:
    if os.path.isfile(os.path.join(path, "adapter_config.json")):
        SFT_ADAPTER_PATH = path
        break

if SFT_ADAPTER_PATH is None:
    print("ERROR: Could not find SFT adapter at any of these locations:")
    for p in _candidates:
        exists = os.path.isdir(p)
        contents = os.listdir(p) if exists else []
        print(f"  {p}  (exists={exists}, files={contents[:5]})")
    print("\nPlease set SFT_ADAPTER_PATH manually below:")
    print('  SFT_ADAPTER_PATH = "/your/path/to/sft"')
else:
    print(f"Found SFT adapter at: {SFT_ADAPTER_PATH}")

# Uncomment to override manually:
# SFT_ADAPTER_PATH = "/content/drive/MyDrive/math-rl-tuning/sft"

## 4. Run GRPO Training

In [ ]:
from math_rl_tuning.grpo_trainer import run_grpo_training

trainer, model, tokenizer = run_grpo_training(
    cfg,
    sft_adapter_path=SFT_ADAPTER_PATH,
    save_to_drive=True,
)

## 5. Quick Sanity Check

In [ ]:
from math_rl_tuning.inference import generate

questions = [
    "What is 15% of 240?",
    "Solve for x: 3x + 7 = 22",
    "A rectangle has length 12 cm and width 5 cm. What is its area?",
]

for q in questions:
    print(f"Q: {q}")
    response = generate(q, model, tokenizer)
    print(f"A: {response[:300]}")
    print("-" * 40)

## 6. Cleanup

In [ ]:
from math_rl_tuning.utils import clean_memory

del model, trainer
clean_memory()